# 04 · Burgers' Equation — The Classic PINN Benchmark

**Level:** Intermediate
**Prerequisites:** `03_heat_equation`

Burgers' equation is *the* canonical PINN test problem — it is the headline
example in Raissi, Perdikaris & Karniadakis (2019). It is **nonlinear** and, at
low viscosity, develops a sharp internal layer (a near-shock) that is genuinely
hard for neural networks. If you can solve Burgers well, you understand the core
of forward PINNs.

### What you will learn
1. Encoding a **nonlinear** convective term `u·∂u/∂x` in the residual
2. Handling a solution that becomes steep (the spectral-bias pain point)
3. Optional **L-BFGS** fine-tuning after Adam — the standard PINN recipe
4. Comparing against a reference solution

### References
- Raissi, Perdikaris & Karniadakis (2019), *J. Comput. Phys.* 378:686-707 (the source of this exact setup)
- Krishnapriyan et al. (2021), *Characterizing possible failure modes in PINNs*, NeurIPS


## 1 · Problem statement (Raissi et al. 2019)

$$\frac{\partial u}{\partial t} + u\,\frac{\partial u}{\partial x}
= \nu\,\frac{\partial^2 u}{\partial x^2},
\qquad x\in[-1,1],\; t\in[0,1],\; \nu=\tfrac{0.01}{\pi}$$

with initial and boundary conditions

$$u(x,0) = -\sin(\pi x),\qquad u(-1,t)=u(1,t)=0.$$

The small viscosity $\nu=0.01/\pi$ lets a steep gradient form near $x=0$ around
$t\approx 0.4$. Capturing that layer is the whole challenge.


In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(1234); np.random.seed(1234)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

nu = 0.01/np.pi


Using device: cpu


In [2]:
class PINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        seq = []
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers)-2:
                seq.append(nn.Tanh())
        self.net = nn.Sequential(*seq)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))

# A deeper/wider net helps resolve the steep layer
model = PINN([2, 40, 40, 40, 40, 40, 1]).to(device)
print("params:", sum(p.numel() for p in model.parameters()))


params: 6721


In [3]:
# --- Sampling ---
N_f, N_0, N_b = 10000, 300, 200

x_f = (torch.rand(N_f,1,device=device)*2 - 1)
t_f = torch.rand(N_f,1,device=device)
x_f.requires_grad_(True); t_f.requires_grad_(True)

x_0 = (torch.rand(N_0,1,device=device)*2 - 1)
t_0 = torch.zeros(N_0,1,device=device)
u_0 = -torch.sin(np.pi*x_0)

t_b = torch.rand(N_b,1,device=device)
x_bL = -torch.ones(N_b,1,device=device)
x_bR =  torch.ones(N_b,1,device=device)

def residual(model, x, t):
    u   = model(x, t)
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx= torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return u_t + u*u_x - nu*u_xx          # <-- nonlinear convective term

def loss_fn():
    r = residual(model, x_f, t_f)
    L_f  = torch.mean(r**2)
    L_0  = torch.mean((model(x_0, t_0) - u_0)**2)
    L_b  = torch.mean(model(x_bL, t_b)**2) + torch.mean(model(x_bR, t_b)**2)
    return L_f + L_0 + L_b


In [4]:
# --- Stage 1: Adam ---
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
ADAM_EPOCHS = 15000
for ep in range(ADAM_EPOCHS):
    opt.zero_grad(); loss = loss_fn(); loss.backward(); opt.step()
    if ep % 2500 == 0:
        print(f"[adam] epoch {ep:5d} | loss {loss.item():.3e}")
print("Adam stage done:", loss.item())


[adam] epoch     0 | loss 4.773e-01
[adam] epoch  2500 | loss 6.369e-03
[adam] epoch  5000 | loss 1.705e-03
[adam] epoch  7500 | loss 7.712e-04
[adam] epoch 10000 | loss 3.686e-04


KeyboardInterrupt: 

In [ ]:
# --- Stage 2: L-BFGS fine-tuning (the standard PINN closer) ---
# L-BFGS is a second-order optimiser that squeezes out the last orders of magnitude.
opt2 = torch.optim.LBFGS(model.parameters(), max_iter=3000, tolerance_grad=1e-9,
                         tolerance_change=1e-12, history_size=50, line_search_fn="strong_wolfe")
def closure():
    opt2.zero_grad(); l = loss_fn(); l.backward(); return l
opt2.step(closure)
print("L-BFGS stage done | final loss:", loss_fn().item())


In [ ]:
# --- Visualise the solution and the forming layer ---
nx, nt = 256, 100
xs = np.linspace(-1, 1, nx); ts = np.linspace(0, 1, nt)
X, Tg = np.meshgrid(xs, ts)
xf = torch.tensor(X.ravel(), dtype=torch.float32, device=device).view(-1,1)
tf = torch.tensor(Tg.ravel(), dtype=torch.float32, device=device).view(-1,1)
with torch.no_grad():
    U = model(xf, tf).cpu().numpy().reshape(nt, nx)

fig, ax = plt.subplots(1, 2, figsize=(13,4))
im = ax[0].pcolormesh(Tg, X, U, shading='auto', cmap='rainbow')
ax[0].set_xlabel('t'); ax[0].set_ylabel('x'); ax[0].set_title('u(x,t) — note the layer near x=0')
fig.colorbar(im, ax=ax[0])
for tv in [0.0, 0.25, 0.5, 0.75]:
    idx = np.argmin(np.abs(ts-tv))
    ax[1].plot(xs, U[idx], lw=2, label=f't={ts[idx]:.2f}')
ax[1].set_xlabel('x'); ax[1].set_ylabel('u'); ax[1].legend()
ax[1].set_title('Snapshots: steepening front')
plt.tight_layout(); plt.show()


## 2 · The two-stage Adam → L-BFGS recipe

This is the single most important practical habit in the PINN literature:

1. **Adam** (first-order, robust) gets you into a good basin.
2. **L-BFGS** (quasi-Newton, uses curvature) drives the loss down several more
   orders of magnitude and sharpens features like the Burgers layer.

Almost every strong PINN result in the literature uses this combination. Adam
alone usually leaves the solution slightly blurry; L-BFGS alone often diverges
from a cold start.
